# 🤖 Práctica Básica con Modelos de Lenguaje Grande (LLMs)

¡Bienvenido a este cuaderno introductorio sobre Modelos de Lenguaje Grande! Aquí aprenderás los conceptos básicos para interactuar con un LLM utilizando la biblioteca `transformers` de Hugging Face.

> 📘 **Versión corregida — 16/11/2026.** Copia de trabajo con las correcciones documentadas en
> `ERRORES_ENCONTRADOS.md` (proyecto `Clase_3_revision`). En el código, cada cambio está marcado
> con `# [CORREGIDO]`. El archivo original no se modificó.
>
> **Aclaración conceptual:** `gpt2` (124 M de parámetros) **no** es un "modelo de lenguaje grande"
> en el sentido actual (GPT-4, Llama 3: cientos de miles de millones). Se usa porque corre en CPU
> en segundos y permite ver el mecanismo: predecir el siguiente token.

**Grupo:** `____` · **Integrantes:** `____, ____, ____, ____`

## ¿Qué aprenderemos?
1. **Configuración:** Instalar las herramientas necesarias.
2. **Cargar un Modelo:** Usar la función `pipeline` para cargar un modelo preentrenado.
3. **Generación de Texto:** Pedirle al modelo que complete una frase o responda una pregunta sencilla.
4. **Análisis de Sentimiento:** Clasificar un texto como positivo o negativo.

---
### ⚠️ Nota sobre el entorno
Para que la generación de texto sea más rápida, te recomendamos usar una GPU.
Ve al menú superior: **Entorno de ejecución > Cambiar tipo de entorno de ejecución** y selecciona **GPU** (T4 GPU es suficiente).

## 1. Configuración e Instalación

Primero, necesitamos instalar la biblioteca `transformers` de Hugging Face, que nos facilita enormemente la descarga y el uso de miles de modelos.

In [ ]:
# Instalamos la biblioteca transformers
# [CORREGIDO] %pip instala en el kernel activo (no en el shell) y fija la serie
# 4.x, que es la que documenta este cuaderno. La 5.x ya existe: no fue probada
# con este material, así que se evita el salto de major en clase.
%pip install -q "transformers>=4.44,<5"

## 2. Nuestro primer "Pipeline" (Generación de texto)

La función `pipeline` es la forma más sencilla de usar modelos en Hugging Face. Abstrae todo el código complejo y nos permite enfocarnos en la tarea.

Vamos a crear un generador de texto usando un modelo pequeño y rápido: `gpt2` (puedes probar otros más adelante).

In [ ]:
import torch
from transformers import pipeline

# [CORREGIDO] usa la GPU si está disponible (validá con: Runtime > Cambiar tipo > T4 GPU)
DEVICE = 0 if torch.cuda.is_available() else -1

# Creamos el pipeline para la tarea de "text-generation" (generación de texto)
# Usaremos el modelo gpt2 básico
generador = pipeline('text-generation', model='gpt2', device=DEVICE)

print(f'Dispositivo: {"GPU" if DEVICE == 0 else "CPU"}')

### ¡Vamos a generar texto!

El modelo GPT-2 original está entrenado principalmente en inglés, por lo que responderá mejor en ese idioma. Le daremos un "prompt" (una frase inicial) y dejaremos que continúe.

In [ ]:
# Nuestro texto inicial (prompt)
texto_inicial = "Artificial intelligence is going to"

# [CORREGIDO] max_new_tokens = tokens NUEVOS (antes max_length=50 contaba
# también los del prompt, así que la longitud real de la salida variaba).
resultado = generador(
    texto_inicial,
    max_new_tokens=30,
    num_return_sequences=1,
    do_sample=False,        # greedy: la salida es reproducible
)

# Imprimimos el resultado (prompt + continuación)
print(resultado[0]['generated_text'])

# Y solo la parte generada:
continuacion = resultado[0]['generated_text'][len(texto_inicial):]
print('\n--- solo la continuación ---')
print(continuacion.strip())

## 🔤 Paréntesis: ¿qué es un "token"?

Los modelos no ven letras ni palabras: ven **tokens**, que son fragmentos producidos por un
tokenizer (BPE). Un token puede ser una palabra, parte de una palabra o un signo. Esto explica
por qué `max_new_tokens` es la forma correcta de limitar la longitud: se cuenta en tokens, no en
caracteres.

In [ ]:
from transformers import AutoTokenizer

# [CORREGIDO] se agrega el paso que conecta la demo con el concepto de "token"
tokenizer = AutoTokenizer.from_pretrained('gpt2')

frase = "Artificial intelligence is going to"
ids = tokenizer(frase)['input_ids']

print(f'Frase: {frase!r}')
print(f'Cantidad de tokens: {len(ids)} (para {len(frase.split())} palabras)')
for i, tid in enumerate(ids):
    print(f'  {i}: id={tid:<6} -> {tokenizer.decode([tid])!r}')

print('\n💡 "intelligence" se parte en dos tokens: por eso los modelos "cuentan" tokens, no palabras.')

## 3. Probando un modelo en Español (Análisis de Sentimiento)

Ahora probaremos una tarea diferente: Análisis de Sentimiento (clasificar si un texto es positivo o negativo). Para esto, usaremos un modelo específicamente entrenado en español.

In [ ]:
# Creamos un nuevo pipeline para "sentiment-analysis"
# Usamos un modelo entrenado para español: 'pysentimiento/robertuito-sentiment-analysis'
analizador_sentimiento = pipeline(
    "sentiment-analysis",
    model="pysentimiento/robertuito-sentiment-analysis",
    device=DEVICE,   # [CORREGIDO] mismo criterio que el generador
)

¡Probémoslo con algunas frases en español!

In [ ]:
frases = [
    "¡Me encanta aprender sobre inteligencia artificial! Es fascinante.",
    "El clima de hoy es muy aburrido y triste.",
    "La película estuvo bien, ni fu ni fa."
]

for frase in frases:
    resultado = analizador_sentimiento(frase)
    etiqueta = resultado[0]['label']   # POS / NEG / NEU
    certeza = resultado[0]['score']
    print(f"Frase: '{frase}'")
    print(f"-> Sentimiento: {etiqueta} (Certeza: {certeza:.2f})\n")

## 🎉 ¡Felicidades!

Has ejecutado con éxito modelos de lenguaje grande para dos tareas diferentes. Esto es solo el principio. Los LLMs modernos pueden resumir textos, traducir idiomas, responder preguntas sobre documentos y mucho más.

**Siguientes pasos sugeridos:**
* Cambia el `texto_inicial` en la sección de generación de texto para ver qué inventa el modelo.
* Prueba tus propias frases en el analizador de sentimientos.
* Explorá el [Hugging Face Hub](https://huggingface.co/models) para descubrir otros modelos que puedas cargar en tu función `pipeline`.
* Probá un modelo más grande en la misma pipeline: `pipeline('text-generation', model='gpt2-medium')`, o uno en español como `PlanTL-GOB-ES/roberta-base-bne`.
* **Demo de la clase (Ollama):** correr un LLM local no necesita Colab. Instalá [Ollama](https://ollama.com), ejecutá `ollama run llama3` y consumí su API en `http://localhost:11434`. Es lo que se mostró en la demo 03 del PDF.